In [ ]:

import scanpy as sc
import pandas as pd
from sklearn import metrics
import torch

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA  # sklearn PCA is used because PCA in scanpy is not stable. 
from anndata import AnnData

import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import sys

from MultiST.MultiST_model import MultiST
from MultiST.utils_func import extract_spot_images
import time
import tracemalloc
import MultiST
from MultiST.utils_func import extract_spot_images , build_hybrid_graph
from MultiST.image_process import  EnhancedDualModalMultiST, train_enhanced_dual_modal_MultiST


In [ ]:
data_path = '/home/wang3wa/Spatial/paper/data/DLPFC'
output_path = '/home/wang3wa/Spatial/paper/MultiST/output'

data_names = os.listdir(data_path)
data_names = [i for i in data_names if i.isdigit()]
print(data_names)  #['151669', '151671', '151675', '151672', '151674', '151673', '151670', '151510', '151509', '151507', '151508', '151676']
data_names = [ '151673']
section_id = '151673'  # For testing purpose
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
# for section_id in data_names:
print(f"================ Start Processing {section_id} ======================")
random_seed = 4
MultiST.fix_seed(random_seed)

n_clusters = 5 if section_id in ['151669','151670','151671','151672'] else 7 

dir_out = f'{output_path}/{section_id}'
os.makedirs(dir_out, exist_ok=True)

tracemalloc.start()
start_time = time.time()

# Load data
adata = sc.read_visium(os.path.join(data_path, section_id))
adata.var_names_make_unique()

df_meta = pd.read_csv(os.path.join(data_path, section_id, 'metadata.tsv'), sep='\t')
adata.obs['layer_guess'] = df_meta['layer_guess'].values

adata.layers['count'] = adata.X.toarray()
sc.pp.filter_genes(adata, min_cells=50)
sc.pp.filter_genes(adata, min_counts=10)
sc.pp.normalize_total(adata, target_sum=1e6)
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", layer='count', n_top_genes=2000)
adata = adata[:, adata.var['highly_variable'] == True]
sc.pp.scale(adata)

adata_X = PCA(n_components=200, random_state=42).fit_transform(adata.X)
adata.obsm['X_pca'] = adata_X

In [ ]:
graph_dict = MultiST.graph_construction(adata, 12)
MultiST_net = MultiST.MultiST(
    X=adata.obsm['X_pca'],
graph_dict=graph_dict,
    use_gan=True,         
    gan_w=0.3,             
    mmd_w=0.2,              
    rec_w=10,
    gcn_w=0.1,
    mode='clustering',
    device=device
)
using_dec = True
if using_dec:
    MultiST_net.train_with_dec(preepoch= 400,epochs=500,N=1)
else:
    MultiST_net.train_without_dec(N=1)
MultiST_feat, _, _, _ = MultiST_net.process()

adata.obsm['MultiST'] = MultiST_feat
# MultiST_feat



In [ ]:
from PIL import Image
image_path = f"/home/wang3wa/Spatial/paper/data/DLPFC/{section_id}/spatial/tissue_hires_image.png"
dir_output = Path('/home/wang3wa/Spatial/paper/SEDR-master/MultiST/output')
dir_output.mkdir(parents=True, exist_ok=True)
optimized_model = EnhancedDualModalMultiST(
    trained_MultiST_model=MultiST_net.model,
    num_clusters=n_clusters,
    image_dim=512,
    hidden_dim=256,
    spatial_k=7,
    patch_size=64,
    image_size=224,
    num_workers=8,                     
    sdm_temperature=0.1,
    use_contrastive=True,
    attention_mode='img2gene',#bidirectional
    gene_weight=0.7,
    apply_normalization=True,   
    save_images=True,         
    normalization_strategy='optimal_diverse'
)
   
gene_features =   MultiST_feat
spatial_coords = adata.obsm['spatial']    
library_id = list(adata.uns['spatial'].keys())[0]  
scalefactors = adata.uns['spatial'][library_id]['scalefactors']
scale_factor = scalefactors['tissue_hires_scalef']    

trainer, eval_results, output_path = train_enhanced_dual_modal_MultiST(
    model=optimized_model,
    gene_features=gene_features,
    image_path=image_path,
    spatial_coords=spatial_coords,
    scale_factor=scale_factor,
    epochs=50,
    batch_size=128,
    device=device,
    log_dir=os.path.join(dir_output),                  
    log_name="my_experiment",
    adata=adata,
    save_images=True  
)

In [ ]:
adata

In [ ]:
import os

os.environ["R_HOME"] = "/home/wang3wa/.conda/envs/wei/lib/R"
os.environ["R_PATH"] = "/home/wang3wa/.conda/envs/wei/bin/R"
os.environ["R_LIBS_USER"] = "/home/wang3wa/.conda/envs/wei/lib/R/library"
os.environ["PATH"] = "/home/wang3wa/.conda/envs/wei/bin:" + os.environ["PATH"]
print('start')
MultiST.mclust_R(adata, n_clusters, use_rep='MultiST', key_added='MultiST_mclust')

In [ ]:
sub_adata = adata[~pd.isnull(adata.obs['layer_guess'])]
ARI = metrics.adjusted_rand_score(sub_adata.obs['layer_guess'], sub_adata.obs['MultiST_mclust'])

fig, axes = plt.subplots(1,2,figsize=(4*2, 4))
sc.pl.spatial(adata, color='layer_guess', ax=axes[0], show=False)
sc.pl.spatial(adata, color='MultiST_mclust', ax=axes[1], show=False)
axes[0].set_title('Manual Annotation')
axes[1].set_title('Clustering(gene): (ARI=%.4f)' % ARI)
plt.tight_layout()
plt.show()

In [ ]:
# predicted_labels = eval_results['predicted_labels']
fusion_features = eval_results['fused_features']
fusion_features = adata.obsm['fused_features']
# adata.obs['four_stages_clusters'] = predicted_labels
adata.obsm['optimized_features'] = fusion_features

print('Clustering finished')
print(adata)

adata.obsm['features'] = eval_results['fused_features']
adata.obsm['MultiST_pca'] = PCA(n_components=50).fit_transform(adata.obsm['features']) 

MultiST.mclust_R(adata, n_clusters, use_rep='MultiST_pca', key_added='mclust')

In [ ]:
sub_adata = adata[~pd.isnull(adata.obs['layer_guess'])]
ARI = metrics.adjusted_rand_score(sub_adata.obs['layer_guess'], sub_adata.obs['mclust'])

fig, axes = plt.subplots(1,2,figsize=(4*2, 4))
sc.pl.spatial(adata, color='layer_guess', ax=axes[0], show=False)
sc.pl.spatial(adata, color='mclust', ax=axes[1], show=False)
axes[0].set_title('Manual Annotation')
axes[1].set_title('Clustering(gene+img): (ARI=%.4f)' % ARI)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import euclidean_distances

def weighted_label_propagation_gaussian(
    adata, label_key='mclust', spatial_key='spatial', 
    k=7, anchor_keep_ratio=0.7, sigma=None, n_iter=10, out_key='refined_label'
):
    coords = adata.obsm[spatial_key]
    labels = adata.obs[label_key].astype(str).values
    le = LabelEncoder()
    label_ids = le.fit_transform(labels)
    n_classes = len(le.classes_)
    N = len(labels)

    # 1.  k neighbors
    A = kneighbors_graph(coords, n_neighbors=k, mode='distance', include_self=False)
    dist_matrix = A.toarray()
    dist_matrix[dist_matrix == 0] = np.inf

    # 2.  W_ij
    if sigma is None:
        sigma = np.median(dist_matrix[dist_matrix != np.inf])
    W = np.exp(-(dist_matrix**2) / (sigma**2))
    W[dist_matrix == np.inf] = 0  

    # 3. agree_i
    agree_scores = np.zeros(N)
    for c in range(n_classes):
        mask_c = (label_ids == c)
        agree_scores[mask_c] = (W[mask_c] * (label_ids == c).astype(float)).sum(axis=1)

    # 4. anchor
    anchor_mask = np.zeros(N, dtype=bool)
    for c in range(n_classes):
        mask_c = (label_ids == c)
        scores_c = agree_scores[mask_c]
        cutoff = np.quantile(scores_c, 1 - anchor_keep_ratio)
        anchor_mask[mask_c] = scores_c >= cutoff

    # 5. label propagation
    Y = np.zeros((N, n_classes))
    Y[np.arange(N), label_ids] = 1
    for _ in range(n_iter):
        Y_new = W.dot(Y)
    
        Y_new[anchor_mask] = Y[anchor_mask]
        Y = Y_new / (Y_new.sum(axis=1, keepdims=True) + 1e-9)

    refined_ids = np.argmax(Y, axis=1)
    adata.obs[out_key] = pd.Series(le.inverse_transform(refined_ids), index=adata.obs.index).astype("category")
    # adata.obs[out_key] = le.inverse_transform(refined_ids).astype('category')
    return adata

In [ ]:
adata = weighted_label_propagation_gaussian(
        adata, label_key='mclust', spatial_key='spatial', 
        k=7, anchor_keep_ratio=0.01, sigma=None, n_iter=10, out_key='refined_mclust'
    )

In [ ]:
sub_adata = adata[~pd.isnull(adata.obs['layer_guess'])]
ARI = metrics.adjusted_rand_score(sub_adata.obs['layer_guess'], sub_adata.obs['refined_mclust'])

fig, axes = plt.subplots(1,2,figsize=(4*2, 4))
sc.pl.spatial(adata, color='layer_guess', ax=axes[0], show=False)
sc.pl.spatial(adata, color='refined_mclust', ax=axes[1], show=False)
axes[0].set_title('Manual Annotation')
axes[1].set_title('Clustering(refined): (ARI=%.4f)' % ARI)
plt.tight_layout()
plt.show()